# Part 1 — Experiment 3: Preprocessing Technique 2 (CLAHE + Unsharp Masking) — CheXpert

Completes the 3-experiment Part 1 protocol on **CheXpert** (matching `0_Baseline_CheXpert.ipynb` and `0_Baseline_pre_CheXpert.ipynb`):
- **Experiment 1 (Baseline):** Frozen DINOv2 ViT-B/14 Linear Probe on raw CheXpert (`0_Baseline_CheXpert.ipynb`)
- **Experiment 2 (Preprocessing 1):** Logarithmic Intensity Transform (`0_Baseline_pre_CheXpert.ipynb`)
- **Experiment 3 (Preprocessing 2 — This Notebook):** **CLAHE (`clipLimit=2.0, tileGridSize=(8,8)`) + Gaussian Unsharp Masking Sharpening (`alpha=1.5, beta=-0.5`)**

### CV Curriculum Basis (Weeks 2–5)
- **CLAHE (Histogram-Based Enhancement):** Enhances local contrast within lung fields without over-saturating bright skeletal structures (ribs/spine).
- **Unsharp Masking (Spatial Filtering / Sharpening):** Emphasizes subtle pleural margins, pulmonary vasculature, and consolidation boundaries prior to DINOv2 patch tokenization.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if os.path.basename(os.getcwd()) != "DinoV2_PEFT":
    if not os.path.exists("DinoV2_PEFT"):
        subprocess.run(["git", "clone", "https://github.com/RudhanLodhi/DinoV2_PEFT.git"], check=True)
    os.chdir("DinoV2_PEFT")

subprocess.run(["pip", "install", "-r", "requirements.txt", "opencv-python-headless", "-q"], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft/exp3_pre2_clahe_unsharp_chexpert"
except ImportError:
    CHECKPOINT_DIR = "outputs/exp3_pre2_clahe_unsharp_chexpert"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Preprocessing Technique 2: CLAHE + Unsharp Masking & Visual Comparison

In [ ]:
"""
Define CLAHE + Gaussian Unsharp Masking transform and visualize before/after on CheXpert samples.
"""
import json
import random
import time
from typing import Any
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T
from src.data import build_chexpert_dataloaders
from src.utils.config import load_config

class CLAHEUnsharpTransform:
    """Applies tile-adaptive CLAHE followed by Gaussian Unsharp Masking sharpening."""
    def __init__(self, clip_limit: float = 2.0, tile_grid: tuple[int, int] = (8, 8), sharp_amount: float = 0.5) -> None:
        self.clip_limit = clip_limit
        self.tile_grid = tile_grid
        self.sharp_amount = sharp_amount

    def __call__(self, img: Image.Image) -> Image.Image:
        gray = np.array(img.convert("L"), dtype=np.uint8)
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid)
        enhanced = clahe.apply(gray)
        blurred = cv2.GaussianBlur(enhanced, (5, 5), sigmaX=1.0)
        sharpened = cv2.addWeighted(enhanced, 1.0 + self.sharp_amount, blurred, -self.sharp_amount, 0)
        return Image.fromarray(sharpened).convert("RGB")

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")
pre = cfg["preprocessing"]
clahe_unsharp = CLAHEUnsharpTransform(clip_limit=2.0, tile_grid=(8, 8), sharp_amount=0.5)

train_tf = T.Compose([
    clahe_unsharp,
    T.RandomResizedCrop(pre["image_size"], scale=(0.75, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=pre["normalize_mean"], std=pre["normalize_std"]),
])
val_tf = T.Compose([
    clahe_unsharp,
    T.Resize(int(pre["image_size"] * 1.14)),
    T.CenterCrop(pre["image_size"]),
    T.ToTensor(),
    T.Normalize(mean=pre["normalize_mean"], std=pre["normalize_std"]),
])

train_loader, val_loader, test_loader = build_chexpert_dataloaders(cfg, train_tf, val_tf)
LABEL_NAMES = list(cfg["dataset"]["label_names"])
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

## 2. Train Frozen DINOv2 ViT-B/14 Linear Probe (Identical Settings to Exp 1 & Exp 2)

In [ ]:
"""
Train frozen DINOv2 ViT-B/14 linear probe with CLAHE + Unsharp Masking and compute 4 metrics.
"""
import copy
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.optim import SGD
from src.evaluation.metrics import compute_auroc
from src.models import count_trainable_parameters, freeze_backbone, load_backbone, MultiLabelHead
from src.training.losses import MultiLabelBCELoss

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone, _ = load_backbone()
freeze_backbone(backbone)
head = MultiLabelHead(in_features=768, num_labels=len(LABEL_NAMES))

class LinearProbeModel(nn.Module):
    def __init__(self, b: nn.Module, h: nn.Module) -> None:
        super().__init__()
        self.backbone = b
        self.head = h
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :])

model = LinearProbeModel(backbone, head).to(device)
trainable, total = count_trainable_parameters(model)
print(f"Trainable: {trainable:,} / {total:,}")

pos_counts = torch.zeros(len(LABEL_NAMES), dtype=torch.float64)
total_n = 0
for b in train_loader:
    pos_counts += b["labels"].sum(dim=0).double()
    total_n += b["labels"].shape[0]
pos_weight = ((total_n - pos_counts) / pos_counts.clamp(min=1.0)).clamp(1.0, 5.0).float().to(device)

criterion = MultiLabelBCELoss(pos_weight=pos_weight)
optimizer = SGD(head.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0)
epochs = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

@torch.no_grad()
def evaluate_chex(m, loader):
    m.eval()
    logits_l, labels_l = [], []
    for b in loader:
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            out = m(b["pixel_values"].to(device, non_blocking=True))
        logits_l.append(out.float().cpu().numpy())
        labels_l.append(b["labels"].numpy())
    probs = 1.0 / (1.0 + np.exp(-np.concatenate(logits_l)))
    y = np.concatenate(labels_l).astype(int)
    preds = (probs >= 0.5).astype(int)
    auc = compute_auroc(y, probs, label_names=LABEL_NAMES)
    return {
        "mean_auroc": float(auc["mean_auroc"]),
        "per_label_auroc": {k: float(v) for k, v in auc["per_label_auroc"].items()},
        "macro_f1": float(f1_score(y, preds, average="macro", zero_division=0.0)),
        "macro_precision": float(precision_score(y, preds, average="macro", zero_division=0.0)),
        "macro_recall": float(recall_score(y, preds, average="macro", zero_division=0.0)),
    }

history, best_auc, best_state = [], -1.0, None
for ep in range(1, epochs + 1):
    model.train()
    rl = 0.0
    for b in train_loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss = criterion(model(b["pixel_values"].to(device)), b["labels"].to(device))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        rl += loss.item()
    scheduler.step()
    vm = evaluate_chex(model, val_loader)
    history.append({"epoch": ep, "train_loss": rl / len(train_loader), "val_auroc": vm["mean_auroc"]})
    print(f"Epoch [{ep:02d}/{epochs}] Loss: {rl/len(train_loader):.4f} | Val AUROC: {vm['mean_auroc']:.4f} | Val F1: {vm['macro_f1']:.4f}")
    if vm["mean_auroc"] > best_auc:
        best_auc = vm["mean_auroc"]
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
val_res = evaluate_chex(model, val_loader)
test_res = evaluate_chex(model, test_loader)
print("\nBest Val Metrics:", json.dumps(val_res, indent=2))
print("Held-out Test Metrics:", json.dumps(test_res, indent=2))
with open(f"{CHECKPOINT_DIR}/results.json", "w", encoding="utf-8") as f:
    json.dump({"val": val_res, "test": test_res, "history": history}, f, indent=2)

## 3. Part 1 Three-Experiment Comparison Table & Curves (CheXpert)

In [ ]:
"""
Render training curves and Part 1 Section 7 comparison table (Exp 1 Baseline vs Exp 2 Log vs Exp 3 CLAHE+Unsharp).
"""
import pandas as pd
import seaborn as sns

part1_summary = pd.DataFrame([
    {"Experiment": "Exp 1: Baseline (0_Baseline_CheXpert)", "Val Mean AUROC": 0.7698, "Base Paper Delta": f"{0.7698 - 0.812:+.4f}"},
    {"Experiment": "Exp 2: Preprocessing 1 — Log Transform", "Val Mean AUROC": 0.7547, "Base Paper Delta": f"{0.7547 - 0.812:+.4f}"},
    {"Experiment": "Exp 3: Preprocessing 2 — CLAHE + Unsharp", "Val Mean AUROC": round(val_res["mean_auroc"], 4), "Base Paper Delta": f"{val_res['mean_auroc'] - 0.812:+.4f}"},
])
print(part1_summary.to_string(index=False))

hdf = pd.DataFrame(history)
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(hdf["epoch"], hdf["train_loss"], marker="o")
axes[0].set_title("Exp 3 (CLAHE + Unsharp) Train Loss — CheXpert")
axes[1].plot(hdf["epoch"], hdf["val_auroc"], marker="s", color="darkorange")
axes[1].axhline(0.7698, color="gray", linestyle="--", label="Exp 1 Baseline (0.7698)")
axes[1].axhline(0.8120, color="crimson", linestyle="--", label="Base Paper (0.8120)")
axes[1].set_title("Exp 3 (CLAHE + Unsharp) Val AUROC — CheXpert")
axes[1].legend()
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/exp3_chexpert_curves.png", dpi=200)
plt.show()